In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, LSTM
from tensorflow.keras import backend as K

# 1. Carga de datos y preparación de etiquetas
df = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv")

mapeo_etiquetas = {'T1': 0, 'T2': 1, 'T3': 2, 'T4': 3}
y = df['t'].map(mapeo_etiquetas)

columnas_a_probar = ['text', 'text_basico', 'text_lema', 'text_pos', 'text_stem_nltk', 'text_lema_nltk']
resultados_dl = []

vocab_size = 5000
max_length = 300 

print("Iniciando evaluación de Redes Neuronales (CNN y LSTM)...")

for col in columnas_a_probar:
    print(f"\nEntrenando modelos para la columna: {col}...")
    
    # Aseguramos que los datos sean strings para evitar errores del Tokenizer
    X = df[col].astype(str)
    
    # 2. División Train/Test (dentro del bucle porque X cambia)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    
    # 3. Tokenización y Padding
    tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train)
    
    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_test_seq = tokenizer.texts_to_sequences(X_test)
    
    X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')
    
    # 4. Pesos de clases
    pesos_clases = compute_class_weight(
        class_weight='balanced', classes=np.unique(y_train), y=y_train
    )
    dict_pesos = dict(enumerate(pesos_clases))

    K.clear_session()
    
    # 5. Construcción y Entrenamiento de la CNN
    modelo_cnn = Sequential([
        Embedding(input_dim=vocab_size, output_dim=100, input_length=max_length),
        Conv1D(filters=128, kernel_size=3, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(4, activation='softmax')
    ])
    
    modelo_cnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    # Entrenamos silenciosamente (verbose=0)
    modelo_cnn.fit(X_train_pad, y_train, epochs=10, batch_size=32, 
                   validation_split=0.1, class_weight=dict_pesos, verbose=0)
    
    y_pred_clases_cnn = np.argmax(modelo_cnn.predict(X_test_pad, verbose=0), axis=1)
    score_cnn = f1_score(y_test, y_pred_clases_cnn, average='macro')

    # ==========================================
    # 6. Construcción y Entrenamiento de la LSTM
    # ==========================================
    modelo_lstm = Sequential([
        Embedding(input_dim=vocab_size, output_dim=100, input_length=max_length),
        LSTM(64, dropout=0.2, recurrent_dropout=0.2),
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(4, activation='softmax')
    ])
    
    modelo_lstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    modelo_lstm.fit(X_train_pad, y_train, epochs=10, batch_size=32, 
                    validation_split=0.1, class_weight=dict_pesos, verbose=0)
    
    y_pred_clases_lstm = np.argmax(modelo_lstm.predict(X_test_pad, verbose=0), axis=1)
    score_lstm = f1_score(y_test, y_pred_clases_lstm, average='macro')
    
    # ==========================================
    # 7. Guardar resultados
    # ==========================================
    resultados_dl.append({
        'Preprocesamiento': col, 
        'Macro-F1 (CNN)': score_cnn,
        'Macro-F1 (LSTM)': score_lstm
    })

# 8. Presentación de resultados finales
df_resultados_dl = pd.DataFrame(resultados_dl).sort_values(by='Macro-F1 (CNN)', ascending=False)

print("\n" + "="*60)
print(" RESULTADOS FINALES DE DEEP LEARNING (Macro-F1)")
print("="*60)
print(df_resultados_dl.to_string(index=False))

Iniciando evaluación de Redes Neuronales (CNN y LSTM)...

Entrenando modelos para la columna: text...



c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Entrenando modelos para la columna: text_basico...


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Entrenando modelos para la columna: text_lema...


...


Entrenando modelos para la columna: text_stem_nltk...


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Entrenando modelos para la columna: text_lema_nltk...


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



 RESULTADOS FINALES DE DEEP LEARNING (Macro-F1)
Preprocesamiento  Macro-F1 (CNN)  Macro-F1 (LSTM)
            text        0.717550         0.314570
       text_lema        0.626806         0.277238
  text_stem_nltk        0.624580         0.304474
        text_pos        0.623142         0.250291
     text_basico        0.616879         0.271169
  text_lema_nltk        0.616381         0.292400


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

print("🚀 Iniciando el Banco de Pruebas Bi-LSTM...")

# Cargar datos locales
df = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv") 

columnas_a_evaluar = ['text', 'text_basico', 'text_lema', 'text_pos', 'text_stem_nltk', 'text_lema_nltk']
columna_target = 't' 

mapeo_etiquetas = {'T1': 0, 'T2': 1, 'T3': 2, 'T4': 3}
y = df[columna_target].map(mapeo_etiquetas).values

# Hiperparámetros base
VOCAB_SIZE = 10000  
MAX_LEN = 512       
EPOCHS = 5
BATCH_SIZE = 16

resultados_f1 = {}

for col in columnas_a_evaluar:
    
    # Escudo anti-PyArrow (listas nativas de Python)
    X = np.array(df[col].astype(str).tolist(), dtype=object)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
    
    #Se aprende el vocabulario y a cada token se le asocia un índice
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>") # Usamos <pad> como índice 0
    tokenizer.fit_on_texts(X_train) 
    
    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_test_seq = tokenizer.texts_to_sequences(X_test)
    
   
    #Se aplicará padding o truncation sobre el texto tokenizado para asegurar que tiene el tamaño de entrada fijado
    X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
    X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')
    

    model = tf.keras.Sequential([
        # Capa de Entrada: Matriz de Embeddings
        # Se crea una matriz de embeddings 10000 x 128
        tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),
        
        # Procesando los datos de entrada en dos direcciones: desde el inicio hasta el final y desde el final hasta el principio
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False)),
        
        # Capas Densas (Clasificación)
        tf.keras.layers.Dropout(0.5), 
        tf.keras.layers.Dense(64, activation='relu'),
        
        # Capa de Salida: Softmax
        tf.keras.layers.Dense(4, activation='softmax') 
    ])
    
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    model.fit(
        X_train_pad, y_train, 
        epochs=EPOCHS, 
        batch_size=BATCH_SIZE, 
        validation_data=(X_test_pad, y_test),
        verbose=0
    )
    
    
    predicciones_prob = model.predict(X_test_pad)
    predicciones_clase = np.argmax(predicciones_prob, axis=-1)
    
    macro_f1 = f1_score(y_test, predicciones_clase, average='macro')
    resultados_f1[col] = macro_f1
    
    tf.keras.backend.clear_session()


for col, score in sorted(resultados_f1.items(), key=lambda item: item[1], reverse=True):
    print(f" {col.ljust(20)} : {score:.4f} Macro-F1")

🚀 Iniciando el Banco de Pruebas Bi-LSTM...


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 76ms/step


...

33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step

🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
   RANKING DE PREPROCESADO (Bi-LSTM)   
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
✔️ text_lema_nltk       : 0.5468 Macro-F1
✔️ text                 : 0.5289 Macro-F1
✔️ text_lema            : 0.5283 Macro-F1
✔️ text_stem_nltk       : 0.5228 Macro-F1
✔️ text_pos             : 0.5158 Macro-F1
✔️ text_basico          : 0.4760 Macro-F1
